# repetitivePrompts — from raw transcripts to the dev-card number

This notebook walks the **exact production pipeline** that produces this
metric's value on the profile page, starting from raw `~/.claude/projects/**/*.jsonl`
transcripts. Every step prints an interim value so you can see what the extractor
is doing.

**Formula (LaTeX):**

```
\text{rate} = \frac{|\{(u_i, u_{i+1}) : \text{qual}(u_i) \wedge \text{qual}(u_{i+1}) \wedge \text{jacc} > 0.6\}|}{\max(1, |\text{qualifying pairs}|)}
```

**Parts:**

- **R** = `repetitive_pairs` (pairs) — User-message pairs with Jaccard similarity > 0.6.
- **Q** = `qualifying_pairs` (pairs) — Pairs where both messages have at least 50 non-stopword tokens.

**Server aggregator:** `server/web/lib/scorer/metrics/repetitivePrompts.ts`
**Wire fields read:** `sessions[].repetitive_pairs`, `sessions[].qualifying_pairs`


## Step 1 — Discover sessions

List every JSONL transcript under `~/.claude/projects/`.


In [ ]:
from pathlib import Path
import sys, os, json, uuid
# Make the scripts/ package importable when running from notebooks/.
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'per-metric' else Path.cwd()))
from scripts.events import find_sessions

sessions = find_sessions()
print(f'discovered {len(sessions)} JSONL transcripts')
for s in sessions[:5]:
    print(f'  {s.session_id[:12]}  project={s.project_root}  '
          f'first={s.first_ts_ms}  last={s.last_ts_ms}')


## Step 2 — Run the production extractor

`extract(device_id, client_version)` is the **single function** that powers
the production `/conductorscore` upload. It scans all sessions, classifies
minutes as HITL/AFK/Idle, counts tool invocations, detects plan signals,
etc., and returns an `ExtractorOutput` matching the wire format.


In [ ]:
from scripts.extractor import extract

payload = extract(device_id=str(uuid.UUID(int=0)), client_version='notebook')
wire = json.loads(payload.to_json())
print(f'extracted {len(wire["sessions"])} sessions into the wire payload')
print(f'schema_version: {wire["device"]["schema_version"]}')


## Step 3 — Inspect the wire fields this metric reads

Per the registry, this metric reads the following wire fields:

- `sessions[].repetitive_pairs`
- `sessions[].qualifying_pairs`

Below are the per-session values for those fields, plus a small distribution summary.


In [ ]:
WIRE_FIELDS = ["sessions[].repetitive_pairs", "sessions[].qualifying_pairs"]

def read(path: str, session: dict):
    if path.startswith('sessions[].'):
        return session.get(path[len('sessions[].'):])
    if path.startswith('config.'):
        return wire.get('config', {}).get(path[len('config.'):])
    return wire.get(path)

for field in WIRE_FIELDS:
    if field.startswith('config.'):
        print(f'{field}: {read(field, {})!r}')
        continue
    values = [read(field, s) for s in wire['sessions']]
    nonzero = [v for v in values if v not in (0, None, [], False)]
    print(f'{field}: {len(nonzero)} non-zero sessions, top 5: ' + str(
        sorted([v for v in nonzero if isinstance(v, (int, float))], reverse=True)[:5] or nonzero[:5]
    ))


## Step 4 — Apply the server aggregator

The TypeScript aggregator at `server/web/lib/scorer/metrics/repetitivePrompts.ts` consumes the
wire payload via the same `aggregate-one.ts` adapter the debug script uses.
We shell out to it through `npx tsx` and parse the result. The output is
the canonical `{ raw, parts }` shape that the API surfaces and the dev-card
tile renders.


In [ ]:
import subprocess
from scripts.debug_metric.registry import _SERVER_WEB, _node20_path_env

result = subprocess.run(
    ['npx', 'tsx', 'scripts/aggregate-one.ts', 'repetitivePrompts'],
    cwd=_SERVER_WEB,
    input=json.dumps(wire),
    capture_output=True, text=True, check=True,
    env=_node20_path_env(),
)
agg = json.loads(result.stdout)
print(json.dumps(agg, indent=2))


## Step 5 — Format like the dev-card

The profile tile renders the `raw` field with metric-specific formatting.
For `repetitivePrompts` the unit is `%`. Below is the final string the dev-card
would display (matches the value on `/u/<your-handle>` for the most recent upload).


In [ ]:
raw = agg.get('raw')
parts = agg.get('parts', [])
unit = '%'

# Mirror the dev-card formatters (see components/dev-card.tsx).
if raw is None:
    display = '—'
elif isinstance(raw, list):
    display = f'{raw[0]} / {raw[1]} {unit}'.strip()
elif isinstance(raw, (int, float)):
    # Round to 2-3 sig figs for the headline; the modal shows full precision.
    if abs(raw) >= 100:
        display = f'{raw:.0f} {unit}'
    elif abs(raw) >= 1:
        display = f'{raw:.2f} {unit}'
    else:
        display = f'{raw:.3f} {unit}'
else:
    display = f'{raw} {unit}'

print(f'\n=== DEV-CARD VALUE ===\n{display}\n=====================')
print()
print('Sub-metrics (shown in the (i) modal):')
for p in parts:
    print(f'  {p["symbol"]} = {p["value"]:,} {p["unit"]}')


## Where this lands on the page

The number printed by Step 5 is what the dev-card tile renders for `repetitivePrompts`.
Open `https://conductorscore.com/u/<your-handle>` and find the tile matching
selector `[data-tile="repetitive-prompts"]`. Click the **ⓘ** button to see the same parts
breakdown printed above, with each symbol annotated by its `label` and `describe`.

For end-to-end provenance (UI ↔ API ↔ DB ↔ upload ↔ this re-extract), run:

```bash
python -m scripts.debug_metric repetitivePrompts --user <your-handle>
```
